<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300" height="100"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><font size=10>Artificial Intelligence and Machine Learning</center></font>
<center><font size=6>LLMs and Prompt Engineering</center></font>

<center><p float="center">
  <img src="https://images.pexels.com/photos/262918/pexels-photo-262918.jpeg?auto=compress&cs=tinysrgb&w=1260&h=750&dpr=1" width=720/>
</p></center>

<center><font size=6>Restaurant Review Analysis</center></font>

## Problem Statement

### Business Context

In the food industry, customer satisfaction plays a pivotal role in shaping the success of individual outlets and the overall brand. A leading global food aggregator is keen on understanding and improving customer experiences across the diverse range of restaurants it lists on its platform. The company recognizes the significance of customer reviews in gaining insights into service quality, food offerings, and overall satisfaction.

### Problem Definition

Despite the abundance of customer reviews available, the company faces significant challenges in deriving actionable insights from these valuable data sources. The manual analysis of extensive amounts of unstructured text data tends to be time-consuming and non-scalable. The key problems to address include:

- **Unstructured Data Challenge**: Customer reviews are expressed in natural language and an unstructured format, creating difficulties in efficiently extracting meaningful information.

- **Scale of Data**: With numerous restaurants, the company accumulates a substantial volume of reviews. Manually processing this vast amount of data is not scalable and necessitates an automated approach.

- **Customer Sentiment Understanding**: Discerning customer sentiments from reviews, whether positive, negative, or neutral, poses a significant challenge. This understanding is crucial for the company to identify the preferences of different customers and devise strategies for targeted marketing.

### Objective

As a data scientist at the company, you have been provided a sample of the customer review data and asked to created a predictive model to analyze the reviews. The objective is to build a robust sentiment analyzer using a Large Language Model (LLM) that accurately predicts the sentiment of customers from the reviews, thereby enhancing the company's ability to understand customer sentiments at scale, enabling data-driven decision-making, and improving overall customer satisfaction.

## Installing and Importing Necessary Libraries

In [ ]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.45 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.45 --force-reinstall --no-cache-dir -q

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# For downloading the models from HF Hub
!pip install huggingface_hub==0.20.3 pandas==2.2.2 -q

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# Importing library for data manipulation
import pandas as pd

# Function to download the model from the Hugging Face model hub
from huggingface_hub import hf_hub_download

# Importing the Llama class from the llama_cpp module
from llama_cpp import Llama

# Importing the json module
import json

## Import the dataset

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
data = pd.read_csv("restaurant_reviews.csv")

## Data Overview

In [ ]:
# checking the first five rows of the data
data.head()

,restaurant_ID,rating_review,review_full
0,FLV202,5,"Totally in love with the Auro of the place, re..."
1,SAV303,5,Kailash colony is brimming with small cafes no...
2,YUM789,5,Excellent taste and awesome decorum. Must visi...
3,TST101,5,I have visited at jw lough/restourant. There w...
4,EAT456,5,Had a great experience in the restaurant food ...


In [ ]:
# checking the shape of the data
data.shape

(20, 3)

**Observations**

- Data has 20 rows and 3 columns

In [ ]:
# checking for missing values
data.isnull().sum()

,0
restaurant_ID,0
rating_review,0
review_full,0


**Observations**

- There are no missing values in the data

In [ ]:
max_len_review = data.loc[data['review_full'].str.len().idxmax(), 'review_full']
print("The longest review_full is:")
print(max_len_review)

The longest review_full is:
I went to Pluck (Hotel Pullman) on a Friday night. We were a group of 7 adults and 1 infant. There weren’t many people in the restaurant when we reached around 10:20PM and very soon the ones that were there also left. So eventually it was just all of us on one table and after that no one else came. Some person came to our table and introduced himself as Ashish and told us that he would take care of our table. The menus were in a tablet which seemed impressive. We were coming straight from the airport after our journey and were hungry and tired too. We quickly ordered 3 set menus, some food items and some drinks. The service from the very beginning was very slow. Apart from Ashish, we could see about 4-5 more waiters inside the restaurant; but when it came to servicing our table, it was only Ashish who was doing it and couldn’t do it effectively. We kept waiting for our order; we had to call him so many times to remind him to bring our order, particularly the

# Model Building

### Loading the model (Llama)

In [ ]:
model_name_or_path = "TheBloke/Llama-2-13B-chat-GGUF"
model_basename = "llama-2-13b-chat.Q5_K_M.gguf" # the model is in gguf format

In [ ]:
# Using hf_hub_download to download a model from the Hugging Face model hub
# The repo_id parameter specifies the model name or path in the Hugging Face repository
# The filename parameter specifies the name of the file to download
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
lcpp_llm = Llama(
    model_path=model_path,
    n_threads=2,  # CPU cores
    n_batch=256,  # Should be between 1 and n_ctx, consider the amount of VRAM in your GPU.
    # n_gpu_layers=41,  # determines the number of GPU layers to utilize; uncomment this line if GPU is available
    n_ctx=1440,  # Context window
)

llama_model_loader: loaded meta data with 19 key-value pairs and 363 tensors from /root/.cache/huggingface/hub/models--TheBloke--Llama-2-13B-chat-GGUF/snapshots/4458acc949de0a9914c3eab623904d4fe999050a/llama-2-13b-chat.Q5_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 5120
llama_model_loader: - kv   4:                          llama.block_count u32              = 40
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 13824
llama_model_loader: - kv   6:                 llama.rope.dimension_

### Loading the model (Mistral)

In [ ]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [ ]:
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

In [ ]:
llm = Llama(
    model_path=model_path,
    n_threads=2,  # CPU cores
    n_batch=200,  # Should be between 1 and n_ctx, consider the amount of VRAM in your GPU.
    # n_gpu_layers=22,  # determines the number of GPU layers to utilize; uncomment this line if GPU is available
    n_ctx=900,
)

### Defining Model Response Parameters

In [ ]:
def generate_response(model_instance, prompt, params=None):
    # Set standard defaults including all requested parameters
    config = {
        "max_tokens": 1024,
        "temperature": 0.01,
        "top_p": 0.95,
        "repeat_penalty": 1.2,
        "top_k": 50,
        "stop": ["[/INST]", "Q:", "INST", "\n\n"],
        "echo": False,
        "seed": 42
    }

    # Update config if specific params are passed during the call
    if params:
        config.update(params)

    # Calling the model instance with the unpacked configuration
    response = model_instance(
        prompt=prompt,
        **config
    )

    return response["choices"][0]["text"].strip()

- **`max_tokens`**: This parameter **specifies the maximum number of tokens that the model should generate** in response to the prompt.

- **`temperature`**: This parameter **controls the randomness of the generated response**. A higher temperature value will result in a more random response, while a lower temperature value will result in a more predictable response.

- **`top_p`**: This parameter **controls the diversity of the generated response by establishing a cumulative probability cutoff for token selection**. A higher value of top_p will result in a more diverse response, while a lower value will result in a less diverse response.

- **`repeat_penalty`**: This parameter **controls the penalty for repeating tokens in the generated response**. A higher value of repeat_penalty will result in a lower probability of repeating tokens, while a lower value will result in a higher probability of repeating tokens.

- **`top_k`**: This parameter **controls the maximum number of most-likely next tokens to consider** when generating the response at each step.

- **`stop`**: This parameter is a **list of tokens that are used to dynamically stop response generation** whenever the tokens in the list are encountered.

- **`echo`**: This parameter **controls whether the input (prompt) to the model should be returned** in the model response.

- **`seed`**: This parameter **specifies a seed value that helps replicate results**.


### Utility function

In [ ]:
# defining a function to parse the JSON output from the model
def extract_json_data(json_str):
    try:
        # Find the indices of the opening and closing curly braces
        json_start = json_str.find('{')
        json_end = json_str.rfind('}')

        if json_start != -1 and json_end != -1:
            extracted_sentiment = json_str[json_start:json_end + 1]  # Extract the JSON object
            data_dict = json.loads(extracted_sentiment)
            return data_dict
        else:
            print(f"Warning: JSON object not found in response: {json_str}")
            return {}
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return {}

## 1. Sentiment Analysis (Llama)

In [ ]:
# creating a copy of the data
data_1 = data.copy()

In [ ]:
# defining the instructions for the model
instruction_1 = """
    You are an AI analyzing restaurant reviews. Classify the sentiment of the provided review into only one of the following categories:
    - Positive
    - Negative
    - Neutral
"""

In [ ]:
# 1. Define the parameters for this specific task
params_task_1 = {"max_tokens": 1024, "temperature": 0.01}

# 2. Apply the optimized generate_response function
data_1['model_response'] = data_1['review_full'].apply(
    lambda x: generate_response(
        lcpp_llm,
        f"[INST]<<SYS>>\n{instruction_1}\n<</SYS>>\n{x}[/INST]",
        params_task_1
    )
)


llama_print_timings:        load time =    1037.39 ms
llama_print_timings:      sample time =      79.11 ms /   121 runs   (    0.65 ms per token,  1529.54 tokens per second)
llama_print_timings: prompt eval time =    1036.86 ms /   234 tokens (    4.43 ms per token,   225.68 tokens per second)
llama_print_timings:        eval time =    8211.98 ms /   120 runs   (   68.43 ms per token,    14.61 tokens per second)
llama_print_timings:       total time =    9875.30 ms /   354 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =    1037.39 ms
llama_print_timings:      sample time =      54.71 ms /    91 runs   (    0.60 ms per token,  1663.41 tokens per second)
llama_print_timings: prompt eval time =     712.87 ms /   243 tokens (    2.93 ms per token,   340.88 tokens per second)
llama_print_timings:        eval time =    6725.56 ms /    90 runs   (   74.73 ms per token,    13.38 tokens per second)
llama_print_timings:       total time =    7829.65 ms /   333 

In [ ]:
data_1['model_response'].head()

,model_response
0,The sentiment of the review is Positive. The r...
1,"Sure! Based on the review, I would classify th..."
2,"Sure! Based on the review you provided, I woul..."
3,Sure! Here's my analysis of the sentiment in y...
4,"Sure! Based on the review, I would classify th..."


In [ ]:
i = 2
print(data_1.loc[i, 'review_full'])

Excellent taste and awesome decorum. Must visit. Subham Barnwal had given us a great service. One of the best experience.


In [ ]:
print(data_1.loc[i, 'model_response'])

Sure! Based on the review you provided, I would classify the sentiment as Positive. The reviewer mentions "excellent taste" and "awesome decorum," which suggests that they had a positive experience at the restaurant. Additionally, they mention "one of the best experiences," which further reinforces a positive sentiment.


In [ ]:
def extract_sentiment(model_response):
    if 'positive' in model_response.lower():
        return 'Positive'
    elif 'negative' in model_response.lower():
        return 'Negative'
    elif 'neutral' in model_response.lower():
        return 'Neutral'

In [ ]:
# applying the function to the model response
data_1['sentiment'] = data_1['model_response'].apply(extract_sentiment)
data_1['sentiment'].head()

,sentiment
0,Positive
1,Positive
2,Positive
3,None
4,Positive


In [ ]:
data_1['sentiment'].value_counts()

,count
sentiment,
Positive,16
Negative,2
Neutral,1


In [ ]:
final_data_1 = data_1.drop(['model_response'], axis=1)
final_data_1.head()

,restaurant_ID,rating_review,review_full,sentiment
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive
3,TST101,5,I have visited at jw lough/restourant. There w...,None
4,EAT456,5,Had a great experience in the restaurant food ...,Positive


**Observations**

- The model provides long conversational explanations instead of just the requested label i.e., Postitive/Negative/Neutral.

- It repeats phrases from the review to justify its sentiment classification.

- The output format is inconsistent and difficult to parse for automated systems and pass through APIs.

## 1. Sentiment Analysis (Mistral)

In [ ]:
# creating a copy of the data
data_1 = data.copy()

**We are going to use an instruction-tuned Mistral model. Hence, the format of the input to the model varies from that of Llama.**

In [ ]:
# defining the instructions for the model
instruction_1 = """
    You are an AI analyzing restaurant reviews. Classify the sentiment of the provided review into only one of the following categories:
    - Positive
    - Negative
    - Neutral
"""

In [ ]:
# 1. Define the parameters for Task 1 (Mistral)
params_mistral_1 = {"max_tokens": 600, "temperature": 0.01}

# 2. Apply the optimized function using the merged f-string
data_1['model_response'] = data_1['review_full'].apply(
    lambda x: generate_response(
        llm,
        f"Q: {instruction_1}\nReview: {x}\nA:",
        params_mistral_1
    )
)


llama_print_timings:        load time =    2630.81 ms
llama_print_timings:      sample time =      88.05 ms /   151 runs   (    0.58 ms per token,  1715.03 tokens per second)
llama_print_timings: prompt eval time =    5307.50 ms /   214 tokens (   24.80 ms per token,    40.32 tokens per second)
llama_print_timings:        eval time =   45595.21 ms /   150 runs   (  303.97 ms per token,     3.29 tokens per second)
llama_print_timings:       total time =   51579.16 ms /   364 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =    2630.81 ms
llama_print_timings:      sample time =      40.85 ms /    67 runs   (    0.61 ms per token,  1639.99 tokens per second)
llama_print_timings: prompt eval time =    1873.51 ms /   232 tokens (    8.08 ms per token,   123.83 tokens per second)
llama_print_timings:        eval time =   20912.76 ms /    66 runs   (  316.86 ms per token,     3.16 tokens per second)
llama_print_timings:       total time =   23074.65 ms /   298 

In [ ]:
data_1['model_response'].head()

,model_response
0,This review expresses a very positive sentimen...
1,The sentiment expressed in this review is posi...
2,The given review expresses a positive sentimen...
3,Positive
4,This review expresses a positive sentiment tow...


In [ ]:
i = 2
print(data_1.loc[i, 'review_full'])

Excellent taste and awesome decorum. Must visit. Subham Barnwal had given us a great service. One of the best experience.


In [ ]:
print(data_1.loc[i, 'model_response'])

The given review expresses a positive sentiment towards the restaurant, as indicated by words such as "excellent," "awesome," "must visit," "great service," and "one of the best experience."


In [ ]:
def extract_sentiment(model_response):
    if 'positive' in model_response.lower():
        return 'Positive'
    elif 'negative' in model_response.lower():
        return 'Negative'
    elif 'neutral' in model_response.lower():
        return 'Neutral'

In [ ]:
# applying the function to the model response
data_1['sentiment'] = data_1['model_response'].apply(extract_sentiment)
data_1['sentiment'].head()

,sentiment
0,Positive
1,Positive
2,Positive
3,Positive
4,Positive


In [ ]:
data_1['sentiment'].value_counts()

,count
sentiment,
Positive,7
Negative,7
Neutral,6


In [ ]:
final_data_1 = data_1.drop(['model_response'], axis=1)
final_data_1.head()

,restaurant_ID,rating_review,review_full,sentiment
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive
4,EAT456,5,Had a great experience in the restaurant food ...,Positive


**Observations**

- The model includes a summary of the review's tone before providing the sentiment.

- The output format is inconsistent and difficult to parse for automated systems and pass through APIs.

## 2. Sentiment Analysis and Returning Structured Output (Llama)

In [ ]:
# creating a copy of the data
data_2 = data.copy()

In [ ]:
# defining the instructions for the model
instruction_2 = """
    You are an AI analyzing restaurant reviews. Classify the sentiment of the provided review into the following categories:
    - Positive
    - Negative
    - Neutral

    Format the output as a JSON object with a single key-value pair as shown below:
    {"sentiment": "your_sentiment_prediction"}

    Only return the JSON, do not return any other information.
"""

In [ ]:
# 1. (Optional) Define a parameter override to save compute time
params_task_2 = {"max_tokens": 128}

# 2. Apply the optimized function
# We add '{"' at the end of the prompt to force the JSON structure
data_2['model_response'] = data_2['review_full'].apply(
    lambda x: "{" + generate_response(
        lcpp_llm,
        f"[INST]<<SYS>>\n{instruction_2}\n<</SYS>>\n{x}[/INST]\n{{",
        params_task_2
    )
)

Llama.generate: prefix-match hit

llama_print_timings:        load time =    1037.39 ms
llama_print_timings:      sample time =       5.33 ms /    10 runs   (    0.53 ms per token,  1876.88 tokens per second)
llama_print_timings: prompt eval time =     680.73 ms /   254 tokens (    2.68 ms per token,   373.13 tokens per second)
llama_print_timings:        eval time =     674.73 ms /     9 runs   (   74.97 ms per token,    13.34 tokens per second)
llama_print_timings:       total time =    1390.11 ms /   263 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =    1037.39 ms
llama_print_timings:      sample time =       5.32 ms /    10 runs   (    0.53 ms per token,  1878.29 tokens per second)
llama_print_timings: prompt eval time =     692.12 ms /   245 tokens (    2.82 ms per token,   353.98 tokens per second)
llama_print_timings:        eval time =     683.50 ms /     9 runs   (   75.94 ms per token,    13.17 tokens per second)
llama_print_timings:       to

In [ ]:
data_2['model_response'].head()

,model_response
0,"{""sentiment"": ""Positive"" }"
1,"{""sentiment"": ""Positive"" }"
2,"{""sentiment"": ""Positive"" }"
3,"{""sentiment"": ""Positive"" }"
4,"{""sentiment"": ""Positive"" }"


In [ ]:
i = 3
print(data_2.loc[i, 'review_full'])

I have visited at jw lough/restourant. There were a first class service at lough, specially Ms.laxmi  who were superbed for handling the client need, me and my family lots enjoyed her specialty in the manner, and Laxmi is a very very good in the client service, I hope when I will come against I would definitely serve from Ms. Laxmi and she is wonderful girl in that service. See you again Ms. Laxmi for the your best service which I have received from you at jw lough/resourant. Thank you JW Marriott Hotel at Atrocity, Delhi


In [ ]:
print(data_2.loc[i, 'model_response'])

{"sentiment": "Positive" }


In [ ]:
# applying the function to the model response
data_2['model_response_parsed'] = data_2['model_response'].apply(extract_json_data)
data_2['model_response_parsed'].head()

,model_response_parsed
0,{'sentiment': 'Positive'}
1,{'sentiment': 'Positive'}
2,{'sentiment': 'Positive'}
3,{'sentiment': 'Positive'}
4,{'sentiment': 'Positive'}


In [ ]:
model_response_parsed_df_2 = pd.json_normalize(data_2['model_response_parsed'])
model_response_parsed_df_2.head()

,sentiment
0,Positive
1,Positive
2,Positive
3,Positive
4,Positive


In [ ]:
data_with_parsed_model_output_2 = pd.concat([data_2, model_response_parsed_df_2], axis=1)
data_with_parsed_model_output_2.head()

,restaurant_ID,rating_review,review_full,model_response,model_response_parsed,sentiment
0,FLV202,5,"Totally in love with the Auro of the place, re...","{""sentiment"": ""Positive"" }",{'sentiment': 'Positive'},Positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,"{""sentiment"": ""Positive"" }",{'sentiment': 'Positive'},Positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,"{""sentiment"": ""Positive"" }",{'sentiment': 'Positive'},Positive
3,TST101,5,I have visited at jw lough/restourant. There w...,"{""sentiment"": ""Positive"" }",{'sentiment': 'Positive'},Positive
4,EAT456,5,Had a great experience in the restaurant food ...,"{""sentiment"": ""Positive"" }",{'sentiment': 'Positive'},Positive


In [ ]:
final_data_2 = data_with_parsed_model_output_2.drop(['model_response','model_response_parsed'], axis=1)
final_data_2.head()

,restaurant_ID,rating_review,review_full,sentiment
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive
4,EAT456,5,Had a great experience in the restaurant food ...,Positive


In [ ]:
final_data_2['sentiment'].value_counts()

,count
sentiment,
Negative,7
Positive,6
neutral,5
Neutral,2


**Observations**

- The model successfully switches from phrases to a clean JSON format.

- It occasionally ignores the exact casing for labels like "Neutral" vs "neutral."

- It demonstrates that negative constraints like "Only return JSON" effectively reduce noise.

## 3. Identifying Overall Sentiment and Sentiment of Aspects of the Experience (Llama)

In [ ]:
# creating a copy of the data
data_3 = data.copy()

In [ ]:
# defining the instructions for the model
instruction_3 = """
    You are an AI analyzing restaurant reviews. Classify the overall sentiment of the provided review into the following categories:
    - "Positive"
    - "Negative"
    - "Neutral"

    Once that is done, check for a mention of the following aspects in the review and classify the sentiment of each aspect as "Positive", "Negative", or "Neutral":
    1. "Food Quality"
    2. "Service"
    3. "Ambience"

    Output the overall sentiment and sentiment for each category in a JSON format with the following keys:
    {
        "Overall": "your_sentiment_prediction",
        "Food Quality": "your_sentiment_prediction",
        "Service": "your_sentiment_prediction",
        "Ambience": "your_sentiment_prediction"
    }

    In case one of the three aspects is not mentioned in the review, set "Not Applicable" (including quotes) for the corresponding JSON key value.

    Only return the JSON, do not return any other information.
"""

In [ ]:
# Define specific parameters for Task 4
params_task_3 = {
    "max_tokens": 1024,
    "temperature": 0.01
}

# Updated Task 3 with JSON forcing
data_3['model_response'] = data_3['review_full'].apply(
    lambda x: "{" + generate_response(
        lcpp_llm,
        f"[INST]<<SYS>>\n{instruction_3}\n<</SYS>>\n{x}[/INST]\n{{",
        params_task_3
    )
)

Llama.generate: prefix-match hit

llama_print_timings:        load time =    1037.39 ms
llama_print_timings:      sample time =      22.64 ms /    43 runs   (    0.53 ms per token,  1899.63 tokens per second)
llama_print_timings: prompt eval time =     629.63 ms /   175 tokens (    3.60 ms per token,   277.94 tokens per second)
llama_print_timings:        eval time =    3129.96 ms /    42 runs   (   74.52 ms per token,    13.42 tokens per second)
llama_print_timings:       total time =    3903.41 ms /   217 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =    1037.39 ms
llama_print_timings:      sample time =      26.23 ms /    43 runs   (    0.61 ms per token,  1639.03 tokens per second)
llama_print_timings: prompt eval time =     683.74 ms /   245 tokens (    2.79 ms per token,   358.32 tokens per second)
llama_print_timings:        eval time =    3078.20 ms /    42 runs   (   73.29 ms per token,    13.64 tokens per second)
llama_print_timings:       to

In [ ]:
data_3['model_response'].head()

,model_response
0,"{""Overall"": ""Positive"",\n""Food Quality"": ""Posi..."
1,"{""Overall"": ""Positive"",\n""Food Quality"": ""Posi..."
2,"{""Overall"": ""Positive"",\n""Food Quality"": ""Posi..."
3,"{""Overall"": ""Positive"",\n""Food Quality"": ""Not ..."
4,"{""Overall"": ""Positive"",\n""Food Quality"": ""Posi..."


In [ ]:
i = 2
print(data_3.loc[i, 'review_full'])

Excellent taste and awesome decorum. Must visit. Subham Barnwal had given us a great service. One of the best experience.


In [ ]:
print(data_3.loc[i, 'model_response'])

{"Overall": "Positive",
"Food Quality": "Positive",
"Service": "Positive",
"Ambience": "Positive"
}


In [ ]:
# applying the function to the model response
data_3['model_response_parsed'] = data_3['model_response'].apply(extract_json_data)
data_3['model_response_parsed'].head()

,model_response_parsed
0,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
1,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
2,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
3,"{'Overall': 'Positive', 'Food Quality': 'Not A..."
4,"{'Overall': 'Positive', 'Food Quality': 'Posit..."


In [ ]:
model_response_parsed_df_3 = pd.json_normalize(data_3['model_response_parsed'])
model_response_parsed_df_3.head()

,Overall,Food Quality,Service,Ambience
0,Positive,Positive,Not Applicable,Positive
1,Positive,Positive,Not Applicable,Positive
2,Positive,Positive,Positive,Positive
3,Positive,Not Applicable,Positive,Not Applicable
4,Positive,Positive,Positive,Not Applicable


In [ ]:
data_with_parsed_model_output_3 = pd.concat([data_3, model_response_parsed_df_3], axis=1)
data_with_parsed_model_output_3.head()

,restaurant_ID,rating_review,review_full,model_response,model_response_parsed,Overall,Food Quality,Service,Ambience
0,FLV202,5,"Totally in love with the Auro of the place, re...","{""Overall"": ""Positive"",\n""Food Quality"": ""Posi...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Not Applicable,Positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,"{""Overall"": ""Positive"",\n""Food Quality"": ""Posi...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Not Applicable,Positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,"{""Overall"": ""Positive"",\n""Food Quality"": ""Posi...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Positive,Positive
3,TST101,5,I have visited at jw lough/restourant. There w...,"{""Overall"": ""Positive"",\n""Food Quality"": ""Not ...","{'Overall': 'Positive', 'Food Quality': 'Not A...",Positive,Not Applicable,Positive,Not Applicable
4,EAT456,5,Had a great experience in the restaurant food ...,"{""Overall"": ""Positive"",\n""Food Quality"": ""Posi...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Positive,Not Applicable


In [ ]:
final_data_3 = data_with_parsed_model_output_3.drop(['model_response','model_response_parsed'], axis=1)
final_data_3.head()

,restaurant_ID,rating_review,review_full,Overall,Food Quality,Service,Ambience
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive,Positive,Not Applicable,Positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive,Positive,Not Applicable,Positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive,Positive,Positive,Positive
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive,Not Applicable,Positive,Not Applicable
4,EAT456,5,Had a great experience in the restaurant food ...,Positive,Positive,Positive,Not Applicable


In [ ]:
final_data_3['Overall'].value_counts()

,count
Overall,
Positive,7
Negative,7
Neutral,6


In [ ]:
final_data_3['Food Quality'].value_counts()

,count
Food Quality,
Positive,6
Not Applicable,5
Mixed,3
Neutral,3
Negative,2
Decent,1


In [ ]:
final_data_3['Service'].value_counts()

,count
Service,
Negative,8
Positive,7
Not Applicable,2
Slow,1
Inconsistent,1
Prompt,1


In [ ]:
final_data_3['Ambience'].value_counts()

,count
Ambience,
Not Applicable,11
Positive,7
Cozy,1
Casual,1


**Observations**

- The model accurately identifies the absence of specific topics using "Not Applicable."

- It maintains the requested JSON structure even when multiple keys are involved.

- It occasionally uses unauthorized labels like "Mixed" or "Decent" instead of the three provided.

## 3. Identifying Overall Sentiment and Sentiment of Aspects of the Experience (Mistral)

In [ ]:
# creating a copy of the data
data_3 = data.copy()

**Note:** We have already predicted the sentiment of the review. We can use this information while designing the prompt for this task. This way, it will reduce the computational complexity.

The sentiment is stored in the 'final_data_1' dataframe which is from the TASK 1.

In [ ]:
# defining the instructions for the model
instruction_3 = """
    You are provided a review and it's sentiment.

    Instructions:
    Classify the sentiment of each aspect as either of "Positive", "Negative", or "Neutral" only and not any other for the given review:
    1. "Food Quality"
    2. "Service"
    3. "Ambience"
    In case one of the three aspects is not mentioned in the review, return "Not Applicable" (including quotes) for the corresponding JSON key value.
    Return the output in the format {"Overall": given sentiment input,"Food Quality": "your_sentiment_prediction","Service": "your_sentiment_prediction","Ambience": "your_sentiment_prediction"}

    Only return the JSON, do not return any other information.

"""

In [ ]:
# 1. Define the parameters for Task 3 (Mistral)
params_task_3_mistral = {
    "max_tokens": 800,
    "temperature": 0.01
}

# 2. Apply the optimized function with JSON forcing
# We pass 'llm' (Mistral) and use the Q: / A: format
data_3['model_response'] = final_data_1.apply(
    lambda row: "{" + generate_response(
        llm,
        f"Q: {instruction_3}\nReview: {row['review_full']}\nSentiment: {row['sentiment']}\nA: {{",
        params_task_3_mistral
    ), axis=1
)

Llama.generate: prefix-match hit

llama_print_timings:        load time =    2630.81 ms
llama_print_timings:      sample time =      24.00 ms /    44 runs   (    0.55 ms per token,  1833.03 tokens per second)
llama_print_timings: prompt eval time =    1952.78 ms /   353 tokens (    5.53 ms per token,   180.77 tokens per second)
llama_print_timings:        eval time =   12625.08 ms /    43 runs   (  293.61 ms per token,     3.41 tokens per second)
llama_print_timings:       total time =   14750.46 ms /   396 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =    2630.81 ms
llama_print_timings:      sample time =      26.19 ms /    46 runs   (    0.57 ms per token,  1756.19 tokens per second)
llama_print_timings: prompt eval time =    1717.73 ms /   239 tokens (    7.19 ms per token,   139.14 tokens per second)
llama_print_timings:        eval time =   15021.25 ms /    45 runs   (  333.81 ms per token,     3.00 tokens per second)
llama_print_timings:       to

In [ ]:
data_3['model_response'].values

array(['{"Overall": "Positive",\n\t"Food Quality": "Positive",\n\t"Service": "Positive",\n\t"Ambience": "Positive"\n}',
       '{"Overall": "Positive",\n\t"Food Quality": "Positive",\n\t"Service": "Not Applicable",\n\t"Ambience": "Positive"\n}',
       '{"Overall": "Positive",\n\t"Food Quality": "Positive",\n\t"Service": "Positive",\n\t"Ambience": "Positive"\n}',
       '{"Overall": "Positive",\n\t"Food Quality": "Not Applicable",\n\t"Service": "Positive",\n\t"Ambience": "Not Applicable"\n}',
       '{"Overall": "Positive",\n\t"Food Quality": "Positive",\n\t"Service": "Positive",\n\t"Ambience": "Not Applicable"\n}',
       '{"Overall": "Positive",\n\t"Food Quality": "Positive",\n\t"Service": "Positive",\n\t"Ambience": "Positive"\n}',
       '{"Overall": "Positive",\n\t"Food Quality": "Neutral",\n\t"Service": "Positive",\n\t"Ambience": "Positive"\n}',
       '{"Overall": "Neutral",\n\t"Food Quality": "Neutral",\n\t"Service": "Negative",\n\t"Ambience": "Not Applicable"\n}',
       '{"Ove

In [ ]:
i = 2
print(data_3.loc[i, 'review_full'])

Excellent taste and awesome decorum. Must visit. Subham Barnwal had given us a great service. One of the best experience.


In [ ]:
print(data_3.loc[i, 'model_response'])

{"Overall": "Positive",
	"Food Quality": "Positive",
	"Service": "Positive",
	"Ambience": "Positive"
}


In [ ]:
# applying the function to the model response
data_3['model_response_parsed'] = data_3['model_response'].apply(extract_json_data)
data_3['model_response_parsed']

,model_response_parsed
0,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
1,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
2,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
3,"{'Overall': 'Positive', 'Food Quality': 'Not A..."
4,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
5,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
6,"{'Overall': 'Positive', 'Food Quality': 'Neutr..."
7,"{'Overall': 'Neutral', 'Food Quality': 'Neutra..."
8,"{'Overall': 'Neutral', 'Food Quality': 'Neutra..."
9,"{'Overall': 'Neutral', 'Food Quality': 'Neutra..."


In [ ]:
model_response_parsed_df_3 = pd.json_normalize(data_3['model_response_parsed'])
model_response_parsed_df_3

,Overall,Food Quality,Service,Ambience
0,Positive,Positive,Positive,Positive
1,Positive,Positive,Not Applicable,Positive
2,Positive,Positive,Positive,Positive
3,Positive,Not Applicable,Positive,Not Applicable
4,Positive,Positive,Positive,Not Applicable
5,Positive,Positive,Positive,Positive
6,Positive,Neutral,Positive,Positive
7,Neutral,Neutral,Negative,Not Applicable
8,Neutral,Neutral,Negative,Positive
9,Neutral,Neutral,Positive,Not Applicable


In [ ]:
model_response_parsed_df_3 = model_response_parsed_df_3.apply(lambda x: x.astype(str).str.lower())

In [ ]:
data_with_parsed_model_output_3 = pd.concat([data_3, model_response_parsed_df_3], axis=1)
data_with_parsed_model_output_3.head()

,restaurant_ID,rating_review,review_full,model_response,model_response_parsed,Overall,Food Quality,Service,Ambience
0,FLV202,5,"Totally in love with the Auro of the place, re...","{""Overall"": ""Positive"",\n\t""Food Quality"": ""Po...","{'Overall': 'Positive', 'Food Quality': 'Posit...",positive,positive,positive,positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,"{""Overall"": ""Positive"",\n\t""Food Quality"": ""Po...","{'Overall': 'Positive', 'Food Quality': 'Posit...",positive,positive,not applicable,positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,"{""Overall"": ""Positive"",\n\t""Food Quality"": ""Po...","{'Overall': 'Positive', 'Food Quality': 'Posit...",positive,positive,positive,positive
3,TST101,5,I have visited at jw lough/restourant. There w...,"{""Overall"": ""Positive"",\n\t""Food Quality"": ""No...","{'Overall': 'Positive', 'Food Quality': 'Not A...",positive,not applicable,positive,not applicable
4,EAT456,5,Had a great experience in the restaurant food ...,"{""Overall"": ""Positive"",\n\t""Food Quality"": ""Po...","{'Overall': 'Positive', 'Food Quality': 'Posit...",positive,positive,positive,not applicable


In [ ]:
final_data_3 = data_with_parsed_model_output_3.drop(['model_response','model_response_parsed'], axis=1)
final_data_3.head()

,restaurant_ID,rating_review,review_full,Overall,Food Quality,Service,Ambience
0,FLV202,5,"Totally in love with the Auro of the place, re...",positive,positive,positive,positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,positive,positive,not applicable,positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,positive,positive,positive,positive
3,TST101,5,I have visited at jw lough/restourant. There w...,positive,not applicable,positive,not applicable
4,EAT456,5,Had a great experience in the restaurant food ...,positive,positive,positive,not applicable


In [ ]:
final_data_3['Overall'].value_counts()

,count
Overall,
positive,7
negative,7
neutral,6


In [ ]:
final_data_3['Food Quality'].value_counts()

,count
Food Quality,
positive,6
neutral,6
not applicable,4
negative,3
decent,1


**Note:** One of the sentiment is 'if not exceptional'. This is most likely positive.

In [ ]:
final_data_3['Service'].value_counts()

,count
Service,
negative,10
positive,8
not applicable,1
prompt,1


In [ ]:
final_data_3['Ambience'].value_counts()

,count
Ambience,
positive,8
not applicable,7
negative,3
casual,1
neutral,1


**Observations**

- The model frequently introduces new, unapproved labels like "prompt" or "casual."

- It struggles with JSON syntax, sometimes including extra newline characters or tabs.

- It shows difficulty in distinguishing between "Neutral" and "Not Applicable" for missing aspects.

- These results highlight the performance constraints of the 7B Mistral model when compared to the higher-capacity 13B LLaMA model.

## 4. Identifying Overall Sentiment, Sentiment of Aspects of the Experience, and the Liked/Disliked Features of the Different Aspects of the Experience (Llama)

In [ ]:
# creating a copy of the data
data_4 = data.copy()

In [ ]:
# defining the instructions for the model
instruction_4 = """
    You are an AI tasked with analyzing restaurant reviews. Your goal is to classify the overall sentiment of the provided review into the following categories:
        - Positive
        - Negative
        - Neutral

    Subsequently, assess the sentiment of specific aspects mentioned in the review, namely:
        1. Food quality
        2. Service
        3. Ambience

    Further, identify liked and/or disliked features associated with each aspect in the review.

    Return the output in the specified JSON format, ensuring consistency and handling missing values appropriately:

    {
        "Overall": "your_sentiment_prediction",
        "Food Quality": "your_sentiment_prediction",
        "Service": "your_sentiment_prediction",
        "Ambience": "your_sentiment_prediction",
        "Food Quality Features": ["liked/disliked features"],
        "Service Features": ["liked/disliked features"],
        "Ambience Features": ["liked/disliked features"]
    }

    The sentiment prediction for Overall, Food Quality, Service, and Ambience should be one of "Positive", "Negative", or "Neutral" only.
    In case one of the three aspects is not mentioned in the review, set "Not Applicable" (including quotes) in the corresponding JSON key value for the sentiment.
    In case there are no liked/disliked features for a particular aspect, assign an empty list in the corresponding JSON key value for the aspect.

    Only return the JSON, do NOT return any other text or information.
"""

In [ ]:
# Define specific parameters for Task 4
params_task_4 = {
    "max_tokens": 1024,
    "temperature": 0.01
}

# Update the application
# Updated Task 4 with "Anchor Forcing"
data_4['model_response'] = data_4['review_full'].apply(
    lambda x: '{"Overall":' + generate_response(
        lcpp_llm,
        f"[INST]<<SYS>>\n{instruction_4}\n<</SYS>>\n{x}[/INST]\n{{\"Overall\":",
        params_task_4
    )
)

Llama.generate: prefix-match hit

llama_print_timings:        load time =    1037.39 ms
llama_print_timings:      sample time =      57.17 ms /   102 runs   (    0.56 ms per token,  1784.18 tokens per second)
llama_print_timings: prompt eval time =    1864.08 ms /   560 tokens (    3.33 ms per token,   300.42 tokens per second)
llama_print_timings:        eval time =    7833.13 ms /   101 runs   (   77.56 ms per token,    12.89 tokens per second)
llama_print_timings:       total time =   10055.46 ms /   661 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =    1037.39 ms
llama_print_timings:      sample time =      55.24 ms /    92 runs   (    0.60 ms per token,  1665.37 tokens per second)
llama_print_timings: prompt eval time =     745.58 ms /   248 tokens (    3.01 ms per token,   332.63 tokens per second)
llama_print_timings:        eval time =    7106.52 ms /    91 runs   (   78.09 ms per token,    12.81 tokens per second)
llama_print_timings:       to

In [ ]:
i = 2
print(data_4.loc[i, 'review_full'])

Excellent taste and awesome decorum. Must visit. Subham Barnwal had given us a great service. One of the best experience.


In [ ]:
print(data_4.loc[i, 'model_response'])

{"Overall":"Positive", 
"Food Quality": "Positive", 
"Service": "Positive", 
"Ambience": "Positive", 
"Food Quality Features": ["Excellent taste"], 
"Service Features": ["great service"], 
"Ambience Features": ["awesome decorum"]}


In [ ]:
# applying the function to the model response
data_4['model_response_parsed'] = data_4['model_response'].apply(extract_json_data)
data_4['model_response_parsed'].head()

,model_response_parsed
0,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
1,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
2,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
3,"{'Overall': 'Positive', 'Food Quality': 'Not A..."
4,"{'Overall': 'Positive', 'Food Quality': 'Posit..."


In [ ]:
data_4[data_4.model_response_parsed == {}]

,restaurant_ID,rating_review,review_full,model_response,model_response_parsed


- There are three model responses that the JSON parser function could not parse
- We'll manually add the values for these three responses

In [ ]:
print(data_4.loc[3, 'model_response'])

{"Overall":"Positive", 
"Food Quality": "Not Applicable", 
"Service": "Positive", 
"Ambience": "Not Applicable", 
"Food Quality Features": [], 
"Service Features": ["Ms. Laxmi's excellent client service"], 
"Ambience Features": []}


In [ ]:
print(data_4.loc[6, 'model_response'])

{"Overall":"Positive", 
"Food Quality": "Mixed", 
"Service": "Positive", 
"Ambience": "Positive", 
"Food Quality Features": ["marinated with too much balsamic vinegar"], 
"Service Features": [], 
"Ambience Features": ["cozy feel", "soft lighting"]}


In [ ]:
print(data_4.loc[7, 'model_response'])

{"Overall":"Neutral", 
"Food Quality": "Mixed", 
"Service": "Neutral", 
"Ambience": "Not Applicable", 
"Food Quality Features": ["some dishes were tasty, others were just average"], 
"Service Features": ["a few attentive staff members"], 
"Ambience Features": []}


In [ ]:
upd_val_1 = {
    "Overall": "Positive",
    "Food Quality": "Positive",
    "Service": "Positive",
    "Ambience": "Not Applicable",
    "Food Quality Features": [],
    "Service Features": ["excellent service"],
    "Ambience Features": []
}

upd_val_2 = {
    "Overall": "Neutral",
    "Food Quality": "Neutral",
    "Service": "Neutral",
    "Ambience": "Not Applicable",
    "Food Quality Features": ["well prepared"],
    "Service Features": ["slow and inattentive"],
    "Ambience Features": ["interior is friendly", "not intimidating"]
}

upd_val_3 = {
    "Overall": "Neutral",
    "Food Quality": "Positive",
    "Service": "Negative",
    "Ambience": "Positive",
    "Food Quality Features": ["Some tasty, others average"],
    "Service Features": ["Attentive staff", "Slow service"],
    "Ambience Features": []
}

# defining the list of indices to update
idx_list = [3,6,7]
data_4.loc[idx_list, 'model_response_parsed'] = [upd_val_1, upd_val_2, upd_val_3]

**Note**: The values model responses that cannot be parsed correctly by the JSON parser function may vary with execution due to the randomness associated with LLMs. Kindly update as observed when run in your system.

In [ ]:
model_response_parsed_df_4 = pd.json_normalize(data_4['model_response_parsed'])
model_response_parsed_df_4.head()

,Overall,Food Quality,Service,Ambience,Food Quality Features,Service Features,Ambience Features
0,Positive,Positive,Positive,Positive,"[pizza straight from the oven, hummus and pita...",[disposable cutlery],"[quaint and cute, pure and positive]"
1,Positive,Positive,Not Applicable,Positive,"[exquisite taste, freshly made]",[],"[peaceful, plants enhanced its beauty]"
2,Positive,Positive,Positive,Positive,[Excellent taste],[great service],[awesome decorum]
3,Positive,Positive,Positive,Not Applicable,[],[excellent service],[]
4,Positive,Positive,Positive,Not Applicable,[fabulous],"[nice, professional]",[]


In [ ]:
data_with_parsed_model_output_4 = pd.concat([data_4, model_response_parsed_df_4], axis=1)
data_with_parsed_model_output_4.head()

,restaurant_ID,rating_review,review_full,model_response,model_response_parsed,Overall,Food Quality,Service,Ambience,Food Quality Features,Service Features,Ambience Features
0,FLV202,5,"Totally in love with the Auro of the place, re...","{""Overall"":""Positive"", \n""Food Quality"": ""Posi...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Positive,Positive,"[pizza straight from the oven, hummus and pita...",[disposable cutlery],"[quaint and cute, pure and positive]"
1,SAV303,5,Kailash colony is brimming with small cafes no...,"{""Overall"":""Positive"", \n""Food Quality"": ""Posi...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Not Applicable,Positive,"[exquisite taste, freshly made]",[],"[peaceful, plants enhanced its beauty]"
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,"{""Overall"":""Positive"", \n""Food Quality"": ""Posi...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Positive,Positive,[Excellent taste],[great service],[awesome decorum]
3,TST101,5,I have visited at jw lough/restourant. There w...,"{""Overall"":""Positive"", \n""Food Quality"": ""Not ...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Positive,Not Applicable,[],[excellent service],[]
4,EAT456,5,Had a great experience in the restaurant food ...,"{""Overall"":""Positive"", \n""Food Quality"": ""Posi...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Positive,Not Applicable,[fabulous],"[nice, professional]",[]


In [ ]:
final_data_4 = data_with_parsed_model_output_4.drop(['model_response','model_response_parsed'], axis=1)
final_data_4.head()

,restaurant_ID,rating_review,review_full,Overall,Food Quality,Service,Ambience,Food Quality Features,Service Features,Ambience Features
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive,Positive,Positive,Positive,"[pizza straight from the oven, hummus and pita...",[disposable cutlery],"[quaint and cute, pure and positive]"
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive,Positive,Not Applicable,Positive,"[exquisite taste, freshly made]",[],"[peaceful, plants enhanced its beauty]"
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive,Positive,Positive,Positive,[Excellent taste],[great service],[awesome decorum]
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive,Positive,Positive,Not Applicable,[],[excellent service],[]
4,EAT456,5,Had a great experience in the restaurant food ...,Positive,Positive,Positive,Not Applicable,[fabulous],"[nice, professional]",[]


In [ ]:
final_data_4['Overall'].value_counts()

,count
Overall,
Neutral,7
Negative,7
Positive,6


In [ ]:
final_data_4['Food Quality'].value_counts()

,count
Food Quality,
Positive,9
Not Applicable,4
Neutral,3
Mixed,2
Negative,2


In [ ]:
final_data_4['Service'].value_counts()

,count
Service,
Positive,8
Negative,7
Not Applicable,2
Neutral,2
Inconsistent,1


In [ ]:
final_data_4['Ambience'].value_counts()

,count
Ambience,
Not Applicable,11
Positive,7
Cozy,1
Neutral,1


**Observations**

- The model successfully extracts specific text snippets as "Features" for each category.

- It correctly uses empty lists [] when no specific features are mentioned in the text.

- It maintains high accuracy while performing classification and extraction in a single pass.

## 5. Identifying Overall Sentiment, Sentiment of Aspects of the Experience, Liked/Disliked Features of the Different Aspects of the Experience, and Sharing a Response (Llama)

In [ ]:
# creating a copy of the data
data_5 = data.copy()

In [ ]:
instruction_5 = """
You are a sentiment analysis system for restaurant reviews.

Your tasks:

Step 1: Classify the overall sentiment of the review as exactly one of:
"Positive"
"Negative"
"Neutral"

Step 2: For each of the following aspects, determine:
1. Whether the aspect is mentioned
2. If mentioned, classify its sentiment as exactly one of:
   "Positive"
   "Negative"
   "Neutral"
3. If NOT mentioned, return:
   "Not Applicable"

Aspects:
- Food Quality
- Service
- Ambience

Step 3: Extract specific liked or disliked features for each mentioned aspect.
- Return short phrases only.
- Do not generate new information.
- If no features are mentioned for an aspect, return an empty list [].
- If the aspect is "Not Applicable", return an empty list [] for its features.

Step 4: Generate a professional and empathetic customer response:
- Always begin with a thank you.
- If Overall = "Positive" → express appreciation and invite them again.
- If Overall = "Neutral" → thank them and ask how the experience could be improved.
- If Overall = "Negative" → apologize sincerely and mention that the concerns will be addressed.

OUTPUT FORMAT RULES (STRICT):
- Return ONLY valid JSON.
- Do NOT include any explanation or extra text.
- Do NOT include trailing commas.
- All sentiment values must be strings and single valued.
- All feature values must be lists of strings.
- Use double quotes for all keys and string values.
- Ensure valid JSON syntax.

Return output in exactly this structure:

{
    "Overall": "Positive/Negative/Neutral",
    "Food Quality": "Positive/Negative/Neutral/Not Applicable",
    "Service": "Positive/Negative/Neutral/Not Applicable",
    "Ambience": "Positive/Negative/Neutral/Not Applicable",
    "Response": "Full customer response text",
    "Food Quality Features": ["feature1", "feature2"],
    "Service Features": ["feature1", "feature2"],
    "Ambience Features": ["feature1", "feature2"]
}
"""

In [ ]:
# Define parameters for Task 5
params_task_5 = {
    "max_tokens": 1440,
    "temperature": 0,
    "top_p": 1,
}

# Update the application
# Updated Task 5 with "Anchor Forcing"
data_5['model_response'] = data_5['review_full'].apply(
    lambda x: '{"Overall":' + generate_response(
        lcpp_llm,
        f"[INST]<<SYS>>\n{instruction_5}\n<</SYS>>\n{x}[/INST]\n{{\"Overall\":",
        params_task_5
    )
)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     717.01 ms
llama_print_timings:      sample time =      77.55 ms /   147 runs   (    0.53 ms per token,  1895.45 tokens per second)
llama_print_timings: prompt eval time =    2037.39 ms /   751 tokens (    2.71 ms per token,   368.61 tokens per second)
llama_print_timings:        eval time =   11167.86 ms /   146 runs   (   76.49 ms per token,    13.07 tokens per second)
llama_print_timings:       total time =   13799.21 ms /   897 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     717.01 ms
llama_print_timings:      sample time =      86.27 ms /   159 runs   (    0.54 ms per token,  1843.07 tokens per second)
llama_print_timings: prompt eval time =     726.85 ms /   248 tokens (    2.93 ms per token,   341.20 tokens per second)
llama_print_timings:        eval time =   12667.70 ms /   158 runs   (   80.18 ms per token,    12.47 tokens per second)
llama_print_timings:       to

In [ ]:
i = 3
print(data_5.loc[i, 'review_full'])

I have visited at jw lough/restourant. There were a first class service at lough, specially Ms.laxmi  who were superbed for handling the client need, me and my family lots enjoyed her specialty in the manner, and Laxmi is a very very good in the client service, I hope when I will come against I would definitely serve from Ms. Laxmi and she is wonderful girl in that service. See you again Ms. Laxmi for the your best service which I have received from you at jw lough/resourant. Thank you JW Marriott Hotel at Atrocity, Delhi


In [ ]:
print(data_5.loc[i, 'model_response'])

{"Overall":"Positive", "Food Quality": "Not Applicable", "Service": "Positive", "Ambience": "Not Applicable", "Response": "Thank you JW Marriott Hotel at Atrocity, Delhi", "Food Quality Features": [], "Service Features": ["superbed for handling the client need"], "Ambience Features": []}


In [ ]:
# applying the function to the model response
data_5['model_response_parsed'] = data_5['model_response'].apply(extract_json_data)
data_5['model_response_parsed'].head()

,model_response_parsed
0,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
1,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
2,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
3,"{'Overall': 'Positive', 'Food Quality': 'Not A..."
4,"{'Overall': 'Positive', 'Food Quality': 'Posit..."


In [ ]:
model_response_parsed_df_5 = pd.json_normalize(data_5['model_response_parsed'])
model_response_parsed_df_5.head()

,Overall,Food Quality,Service,Ambience,Response,Food Quality Features,Service Features,Ambience Features
0,Positive,Positive,Positive,Positive,Thank you so much for your kind words! We're t...,"[pizza, hummus]",[open kitchen],"[quaint, cute]"
1,Positive,Positive,Not Applicable,Positive,Thank you for your kind words! We're thrilled ...,"[freshly made, Wood Fired oven, exquisite taste]",[],"[peaceful, plants]"
2,Positive,Positive,Positive,Positive,Thank you so much for your kind words! We're t...,[excellent taste],[great service],[awesome decorum]
3,Positive,Not Applicable,Positive,Not Applicable,"Thank you JW Marriott Hotel at Atrocity, Delhi",[],[superbed for handling the client need],[]
4,Positive,Positive,Positive,Not Applicable,Thank you so much for taking the time to share...,[fabulous],"[professional, helpful]",[]


In [ ]:
data_with_parsed_model_output_5 = pd.concat([data_5, model_response_parsed_df_5], axis=1)
data_with_parsed_model_output_5.head()

,restaurant_ID,rating_review,review_full,model_response,model_response_parsed,Overall,Food Quality,Service,Ambience,Response,Food Quality Features,Service Features,Ambience Features
0,FLV202,5,"Totally in love with the Auro of the place, re...","{""Overall"":""Positive"", ""Food Quality"": ""Positi...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Positive,Positive,Thank you so much for your kind words! We're t...,"[pizza, hummus]",[open kitchen],"[quaint, cute]"
1,SAV303,5,Kailash colony is brimming with small cafes no...,"{""Overall"":""Positive"",\n""Food Quality"": ""Posit...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Not Applicable,Positive,Thank you for your kind words! We're thrilled ...,"[freshly made, Wood Fired oven, exquisite taste]",[],"[peaceful, plants]"
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,"{""Overall"":""Positive"",\n""Food Quality"": ""Posit...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Positive,Positive,Thank you so much for your kind words! We're t...,[excellent taste],[great service],[awesome decorum]
3,TST101,5,I have visited at jw lough/restourant. There w...,"{""Overall"":""Positive"", ""Food Quality"": ""Not Ap...","{'Overall': 'Positive', 'Food Quality': 'Not A...",Positive,Not Applicable,Positive,Not Applicable,"Thank you JW Marriott Hotel at Atrocity, Delhi",[],[superbed for handling the client need],[]
4,EAT456,5,Had a great experience in the restaurant food ...,"{""Overall"":""Positive"", ""Food Quality"": ""Positi...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Positive,Not Applicable,Thank you so much for taking the time to share...,[fabulous],"[professional, helpful]",[]


In [ ]:
final_data_5 = data_with_parsed_model_output_5.drop(['model_response','model_response_parsed'], axis=1)
final_data_5.head()

,restaurant_ID,rating_review,review_full,Overall,Food Quality,Service,Ambience,Response,Food Quality Features,Service Features,Ambience Features
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive,Positive,Positive,Positive,Thank you so much for your kind words! We're t...,"[pizza, hummus]",[open kitchen],"[quaint, cute]"
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive,Positive,Not Applicable,Positive,Thank you for your kind words! We're thrilled ...,"[freshly made, Wood Fired oven, exquisite taste]",[],"[peaceful, plants]"
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive,Positive,Positive,Positive,Thank you so much for your kind words! We're t...,[excellent taste],[great service],[awesome decorum]
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive,Not Applicable,Positive,Not Applicable,"Thank you JW Marriott Hotel at Atrocity, Delhi",[],[superbed for handling the client need],[]
4,EAT456,5,Had a great experience in the restaurant food ...,Positive,Positive,Positive,Not Applicable,Thank you so much for taking the time to share...,[fabulous],"[professional, helpful]",[]


In [ ]:
final_data_5['Overall'].value_counts()

,count
Overall,
Neutral,7
Negative,7
Positive,6


In [ ]:
final_data_5['Food Quality'].value_counts()

,count
Food Quality,
Positive,9
Mixed,4
Not Applicable,3
Negative,3
Decent/Not Applicable,1


In [ ]:
final_data_5['Service'].value_counts()

,count
Service,
Positive,6
Negative,6
Not Applicable,3
Good,2
Slow,1
Inconsistent,1
Prompt/Not Applicable,1


In [ ]:
final_data_5['Ambience'].value_counts()

,count
Ambience,
Not Applicable,11
Positive,5
Good,2
Cozy,1
Casual/Not Applicable,1


**Observations**

- The step-by-step instruction format ensures the final response matches the detected sentiment.

- It strictly follows the response generation rules.

- It eliminates all conversational filler, returning only the final structured JSON object.

- But we still see extra lables in aspect based sentiments. Reason being while llama is a 13 billion parameter model, we still need bigger models to do a lot of complex combination of tasks in single pass.

<font size=6 color="navyblue">Power Ahead!</font>
___